# 3.8 — Taxi Trips with the DataFrame API

**Chapter 3, sections 3.2, 3.3 and 3.6**, and the starting point for **Exercise 11**.

**The question this notebook answers:** everything so far has run on tables of three rows built
from Python literals. What does the same vocabulary look like on a real file — headerless, with
seventeen columns, some of them junk?

This is the applied counterpart to [3.1](03.01%20DataFrame%20Basics.ipynb): read a CSV, give it
a schema, name and drop columns, derive new ones, aggregate, sort, and drop back to an RDD where
that is genuinely wanted. Exercise 11 continues from here by writing the result as partitioned
Parquet and measuring the difference, which is the procedure of
[3.4](03.04%20Parquet%20vs%20CSV.ipynb).

**Data.** New York City taxi trips, `taxi-data-sorted-small.csv.bz2` — two million rows,
seventeen columns, no header row, covering nineteen days of January 2013. A `bzip2` file is
splittable, so Spark reads it across all cores rather than in one task.

`taxi-data-sorted-verysmall.csv` sits beside it and is the same seventeen columns, but it is
the first ten thousand rows of a file sorted by time and therefore covers a single hour of a
single day. It is fine for checking that a query runs and useless for anything involving a
date, which is most of this notebook.

Runs on a laptop in well under a minute.

In [1]:
# --- CS-777 session setup ------------------------------------------------
import os, tempfile
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType,
                               IntegerType, DoubleType, TimestampType)

DATA = os.environ.get("CS777_DATA", "../data")          # -> code/data/
SCRATCH = os.environ.get("CS777_SCRATCH", os.path.join(tempfile.gettempdir(), "cs777"))
os.makedirs(SCRATCH, exist_ok=True)

spark = (SparkSession.builder
         .appName("CS777-3.8")
         .master("local[*]")
         .config("spark.ui.showConsoleProgress", "false")   # keep printed output clean
         .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

TAXI = f"{DATA}/taxi-data-sorted-small.csv.bz2"       # 2 M rows, 19 days of Jan 2013
# A 10 000-row sample, all within one hour of 2013-01-01 -- fast, but no date range:
# TAXI = f"{DATA}/taxi-data-sorted-verysmall.csv"
print("Spark", spark.version)
print("reading", TAXI)

Spark 4.2.0
reading ../data/taxi-data-sorted-small.csv.bz2


## Declaring the schema

The file has no header row, so the column names have to come from somewhere; and it has two
million rows here and tens of millions in the full dataset, so `inferSchema` would mean
decompressing and parsing all of it twice before the first query. Section 3.6 argues for declaring the schema, and this is what
that looks like on a real file.

The seventeen columns, in file order, are: two hashed identifiers, two timestamps, the trip's
duration and distance, four coordinates, the payment type, and six money amounts. `payment_type` is mostly `CRD` (card) and `CSH`
(cash), with a handful of other codes.

In [2]:
taxi_schema = StructType([
    StructField("medallion",         StringType(),    True),   # hashed cab identifier
    StructField("hack_license",      StringType(),    True),   # hashed driver identifier
    StructField("pickup_datetime",   TimestampType(), True),
    StructField("dropoff_datetime",  TimestampType(), True),
    StructField("trip_time",         IntegerType(),   True),   # seconds
    StructField("trip_distance",     DoubleType(),    True),   # miles
    StructField("pickup_longitude",  DoubleType(),    True),
    StructField("pickup_latitude",   DoubleType(),    True),
    StructField("dropoff_longitude", DoubleType(),    True),
    StructField("dropoff_latitude",  DoubleType(),    True),
    StructField("payment_type",      StringType(),    True),   # CRD, CSH, and a few others
    StructField("fare_amount",       DoubleType(),    True),
    StructField("surcharge",         DoubleType(),    True),
    StructField("mta_tax",           DoubleType(),    True),
    StructField("tip_amount",        DoubleType(),    True),
    StructField("tolls_amount",      DoubleType(),    True),
    StructField("total_amount",      DoubleType(),    True),
])

df = (spark.read
      .schema(taxi_schema)                 # declared: one pass over the file
      .option("header", "false")
      .csv(TAXI))

print("rows:", df.count())
df.printSchema()

rows: 1999999
root
 |-- medallion: string (nullable = true)
 |-- hack_license: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- trip_time: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- surcharge: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)



In [3]:
df.select("pickup_datetime", "trip_time", "trip_distance",
          "payment_type", "fare_amount", "total_amount").show(5)

+-------------------+---------+-------------+------------+-----------+------------+
|    pickup_datetime|trip_time|trip_distance|payment_type|fare_amount|total_amount|
+-------------------+---------+-------------+------------+-----------+------------+
|2013-01-01 00:00:00|      120|         0.44|         CSH|        3.5|         4.5|
|2013-01-01 00:02:00|        0|          0.0|         CSH|       27.0|        27.5|
|2013-01-01 00:01:00|      120|         0.71|         CSH|        4.0|         5.0|
|2013-01-01 00:01:00|      120|         0.48|         CSH|        4.0|         5.0|
|2013-01-01 00:01:00|      120|         0.61|         CRD|        4.0|         5.0|
+-------------------+---------+-------------+------------+-----------+------------+
only showing top 5 rows


## Cleaning: the rows that should not be there

Real trip data contains records that are impossible rather than merely unusual: a trip of zero
distance, a fare of zero dollars, a duration of one second. They are not outliers to be
reasoned about; they are artefacts of the meter, and every aggregate computed over them is
wrong. Filtering them is the DataFrame equivalent of the hand-written `correctRows` function
this notebook's predecessor used on the RDD.

In [4]:
clean = df.where(
    (F.col("trip_time")     > 60)     &     # at least a minute
    (F.col("trip_distance") > 0.10)   &     # at least a tenth of a mile
    (F.col("fare_amount")   > 0.10)   &
    (F.col("total_amount")  > 0.10))

print(f"{df.count():,} rows in  ->  {clean.count():,} rows kept "
      f"({100 * clean.count() / df.count():.1f}%)")
clean = clean.cache()                       # two actions above, several below
clean.count()

1,999,999 rows in  ->  1,965,346 rows kept (98.3%)


1965346

## Naming, dropping, deriving

Three operations from section 3.3, on a table wide enough for them to matter. `drop` removes
the four coordinate columns, which nothing here uses. `withColumn` derives the calendar parts of
the pickup timestamp — the columns Exercise 11 will partition by — and a couple of measures the
raw file does not carry.

In [5]:
trips = (clean
         .drop("pickup_longitude", "pickup_latitude",
               "dropoff_longitude", "dropoff_latitude")
         # calendar parts, from the timestamp rather than by slicing the string
         .withColumn("year",  F.year("pickup_datetime"))
         .withColumn("month", F.month("pickup_datetime"))
         .withColumn("day",   F.dayofmonth("pickup_datetime"))
         .withColumn("hour",  F.hour("pickup_datetime"))
         # derived measures
         .withColumn("minutes", F.round(F.col("trip_time") / 60, 1))
         .withColumn("mph",     F.round(F.col("trip_distance") /
                                        (F.col("trip_time") / 3600), 1))
         .withColumn("tip_pct", F.round(100 * F.col("tip_amount") /
                                        F.col("fare_amount"), 1)))

trips.select("pickup_datetime", "year", "month", "day", "hour",
             "minutes", "trip_distance", "mph", "tip_pct").show(5)

+-------------------+----+-----+---+----+-------+-------------+----+-------+
|    pickup_datetime|year|month|day|hour|minutes|trip_distance| mph|tip_pct|
+-------------------+----+-----+---+----+-------+-------------+----+-------+
|2013-01-01 00:00:00|2013|    1|  1|   0|    2.0|         0.44|13.2|    0.0|
|2013-01-01 00:01:00|2013|    1|  1|   0|    2.0|         0.71|21.3|    0.0|
|2013-01-01 00:01:00|2013|    1|  1|   0|    2.0|         0.48|14.4|    0.0|
|2013-01-01 00:01:00|2013|    1|  1|   0|    2.0|         0.61|18.3|    0.0|
|2013-01-01 00:00:00|2013|    1|  1|   0|    4.0|         1.71|25.7|    0.0|
+-------------------+----+-----+---+----+-------+-------------+----+-------+
only showing top 5 rows


The predecessor of this notebook derived `year` and `month` with `substring(pickup_datetime, 1, 4)`,
which works only because the timestamp happens to be rendered as text in ISO order. With the
column declared as a `TimestampType`, `F.year` and `F.month` say what they mean, survive a
change of format, and are the columns a `partitionBy` should be given.

## Aggregating and ordering

`groupBy` with `agg`, then `orderBy` and `limit` for the ubiquitous top-*k* question.

In [6]:
print("trips and revenue by payment type")
(trips.groupBy("payment_type")
      .agg(F.count("*").alias("trips"),
           F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
           F.round(F.avg("tip_pct"), 1).alias("avg_tip_pct"),
           F.sum("total_amount").cast("decimal(14,2)").alias("revenue"))
      .orderBy(F.desc("trips")).show())

trips and revenue by payment type


+------------+------+--------+-----------+-----------+
|payment_type| trips|avg_fare|avg_tip_pct|    revenue|
+------------+------+--------+-----------+-----------+
|         CRD|997878|   12.67|       19.7|16140454.75|
|         CSH|965916|   11.06|        0.0|11625794.06|
|         UNK|  1550|   15.85|       22.0|   31789.53|
|         DIS|     2|    5.25|        0.0|      11.50|
+------------+------+--------+-----------+-----------+



In [7]:
print("the five busiest hours of the day")
(trips.groupBy("hour")
      .agg(F.count("*").alias("trips"),
           F.round(F.avg("mph"), 1).alias("avg_mph"))
      .orderBy(F.desc("trips")).limit(5).show())

the five busiest hours of the day


+----+------+-------+
|hour| trips|avg_mph|
+----+------+-------+
|  18|129586|   12.5|
|  19|128755|   13.5|
|  20|114003|   14.9|
|  17|110219|   12.7|
|  21|108090|   15.5|
+----+------+-------+



In [8]:
print("fare band -> number of trips  (a derived column used as a grouping key)")
(trips.withColumn("fare_band", F.col("fare_amount").cast("int"))
      .groupBy("fare_band").count()
      .orderBy(F.desc("count")).limit(8).show())

fare band -> number of trips  (a derived column used as a grouping key)


+---------+------+
|fare_band| count|
+---------+------+
|        6|219742|
|        7|207448|
|        5|204170|
|        8|184803|
|        9|155105|
|        4|131751|
|       10|128021|
|       11|104035|
+---------+------+



## `describe`, and the danger of `toPandas`

`describe` computes count, mean, standard deviation, minimum and maximum for the numeric columns
and returns the result as itself a DataFrame. Naming the columns explicitly is worth doing on a
wide table: `describe()` with no arguments summarises everything, including the hashed
identifiers, which is slow and not informative.

In [9]:
trips.describe("trip_distance", "minutes", "mph", "fare_amount", "tip_pct").show()

+-------+------------------+------------------+------------------+------------------+------------------+
|summary|     trip_distance|           minutes|               mph|       fare_amount|           tip_pct|
+-------+------------------+------------------+------------------+------------------+------------------+
|  count|           1965346|           1965346|           1965346|           1965346|           1965346|
|   mean| 2.937460930543528|11.418072186780343|14.079662257943456|11.881009679720519|10.040494803459726|
| stddev|3.4490328798807357| 8.054192042596986| 6.937986374955161| 9.708550675677659|12.606922432959843|
|    min|              0.11|               1.0|               0.2|               2.5|               0.0|
|    max|             95.85|             180.0|             877.5|             360.0|            3407.8|
+-------+------------------+------------------+------------------+------------------+------------------+



In [10]:
# toPandas() moves the ENTIRE table onto the driver.  Safe here only because the result
# has already been reduced to a handful of rows -- which is the rule, not the exception.
by_hour = (trips.groupBy("hour").agg(F.count("*").alias("trips")).orderBy("hour").toPandas())
print(type(by_hour))
print(by_hour.to_string(index=False))

<class 'pandas.core.frame.DataFrame'>
 hour  trips
    0  64964
    1  46543
    2  37581
    3  29176
    4  22638
    5  20169
    6  42011
    7  74852
    8  94967
    9  94114
   10  86925
   11  91517
   12  97593
   13  97247
   14 101033
   15  99544
   16  91833
   17 110219
   18 129586
   19 128755
   20 114003
   21 108090
   22 102383
   23  79603


## Dropping back to the RDD

Section 3.2.1 notes that `.rdd` exposes the underlying RDD of `Row` objects, for the occasional
operation with no DataFrame equivalent. It is worth knowing and worth avoiding: crossing back
means giving up the optimizer and paying the Python serialization boundary again, so it should
be done on a result already reduced to a small size, as here.

In [11]:
rows = trips.groupBy("hour").count().rdd
print("as Row objects: ", rows.take(3))
print("as plain tuples:", trips.groupBy("hour").count().rdd.map(tuple).take(3))

as Row objects:  [Row(hour=12, count=97593), Row(hour=1, count=46543), Row(hour=13, count=97247)]
as plain tuples: [(12, 97593), (1, 46543), (13, 97247)]


## Where Exercise 11 continues

> *Read the taxi CSV, declare an explicit schema for it, and write the result as Parquet
> partitioned by year and month. Then run a query that restricts to a single month against each
> of the two formats, and report the bytes read and the wall-clock time for both.*

The schema is declared above and `year` and `month` are derived. The write is one line. **One
warning before you run it**, and it is the reason to read
[3.4](03.04%20Parquet%20vs%20CSV.ipynb) first: the taxi data shipped with this course covers
**January 2013 only**. Partitioning it by year and month therefore produces exactly one
partition, and a filter on that month prunes nothing at all — so the comparison the exercise
asks for would measure column pruning and compression and nothing else, and the third mechanism
would silently contribute zero.

Partition by `day` instead and the exercise works as intended: nineteen directories, of which a
single-day filter prunes eighteen. The mechanism being demonstrated is identical; only the
cardinality of the partitioning column changes.

In [12]:
OUT = os.path.join(SCRATCH, "ch03-taxi")

# `repartition` by the same columns before writing.  Without it every task holds an open
# writer for every partition value it happens to contain -- eighteen tasks times nineteen
# days is 342 open row-group buffers, which is how a 2 M-row write runs a laptop out of
# heap.  Repartitioning first sends each day to one task, which then writes one file.
(trips.repartition("year", "month").write.mode("overwrite")
      .partitionBy("year", "month")
      .parquet(os.path.join(OUT, "by_month")))

# What actually demonstrates partition pruning on this data.
(trips.repartition("day").write.mode("overwrite")
      .partitionBy("day")
      .parquet(os.path.join(OUT, "by_day")))

def partitions(path):
    return sorted(d for d in os.listdir(path) if "=" in d)

print("partitionBy('year','month') ->", partitions(os.path.join(OUT, "by_month")),
      "-- one directory, so a month filter prunes nothing")
print("partitionBy('day')          ->", len(partitions(os.path.join(OUT, "by_day"))),
      "directories, of which a single-day filter reads one")

partitionBy('year','month') -> ['year=2013'] -- one directory, so a month filter prunes nothing
partitionBy('day')          -> 19 directories, of which a single-day filter reads one


In [13]:
# The comparison the exercise asks for, in the form 3.4 uses.  `PartitionFilters` in the plan
# is the evidence that the directories were pruned before any file was opened.
one_day = (spark.read.parquet(os.path.join(OUT, "by_day"))
           .where(F.col("day") == 15)
           .groupBy("payment_type")
           .agg(F.count("*").alias("trips"), F.round(F.avg("fare_amount"), 2).alias("avg_fare")))
one_day.show()
one_day.explain()

+------------+------+--------+
|payment_type| trips|avg_fare|
+------------+------+--------+
|         CSH|106139|   10.61|
|         CRD|134276|   12.42|
|         UNK|   277|   16.45|
+------------+------+--------+

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[payment_type#5713], functions=[count(1), avg(fare_amount#5714)])
   +- Exchange hashpartitioning(payment_type#5713, 200), ENSURE_REQUIREMENTS, [plan_id=1064]
      +- HashAggregate(keys=[payment_type#5713], functions=[partial_count(1), partial_avg(fare_amount#5714)])
         +- Project [payment_type#5713, fare_amount#5714]
            +- FileScan parquet [payment_type#5713,fare_amount#5714,day#5726] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/var/folders/wh/ptq7zytj1gz42sqc0rs52tm40000gn/T/cs777/ch03-taxi/..., PartitionFilters: [isnotnull(day#5726), (day#5726 = 15)], PushedFilters: [], ReadSchema: struct<payment_type:string,fare_amount:double>




In [14]:
clean.unpersist()
print("scratch written to:", OUT)

scratch written to: /var/folders/wh/ptq7zytj1gz42sqc0rs52tm40000gn/T/cs777/ch03-taxi


## Conclusion

The same vocabulary as [3.1](03.01%20DataFrame%20Basics.ipynb), on a file rather than a literal,
and three habits that only show up at that scale:

* **Declare the schema.** On a headerless file it is the only way to get names at all, and it
  saves the extra pass that inference costs. Declaring `pickup_datetime` as a timestamp rather
  than a string is what lets `F.year` and `F.month` replace string slicing.
* **Filter the impossible rows before aggregating.** Zero-distance, zero-fare trips are meter
  artefacts, and every average computed over them is wrong.
* **`toPandas()` after the aggregation, never before.** The rule is the same as for `collect`:
  it is safe on a result already reduced to something a single machine can hold.

For Exercise 11, everything needed is above — with the caveat that this dataset spans one month,
so partition by `day` to see partition pruning actually do something. The measurement procedure
is in [3.4](03.04%20Parquet%20vs%20CSV.ipynb).